# Обучение модели ранжирования

На предыдущих этапах была построена рекомендательная система.

1. Генерация до 100 товаров-кандидатов для каждой сессии;
2. Формирование признаков для каждой пары `сессия — товар-кандидат`.

Сейчас настало время обучить модель ранжирования. Ее задача - определить порядок товаров-кандидатов внутри каждой сессии.

В качестве модели используем `CatBoostRanker`.

Модель получает признаки пары `сессия — товар` и обучается присваивать более высокий score товарам с большей релевантностью:

- `0` — товар не появился в будущем;
- `1` — click;
- `2` — cart;
- `3` — order.

После обучения товары внутри каждой сессии будут отсортированы по предсказанному score, после чего сформируем итоговый Top-20.

In [41]:
import pandas as pd
from pathlib import Path

data_path = Path("../artifacts/ranking_dataset.pkl")

train_df = pd.read_pickle(data_path)

print("Размер:", train_df.shape)
train_df.head()

Размер: (744109, 13)


,item_click_count,item_cart_count,item_order_count,item_total_count,item_recency,item_interaction_span,covisitation_score,item_click_rate,item_cart_rate,item_order_rate,session,aid,target
0,2,0,0,2,0.0,172938592,4,1.000000,0.000000,0.000000,0,661144,0
1,3,0,0,3,14996.0,214214139,0,1.000000,0.000000,0.000000,0,570955,0
2,3,1,0,4,44215.0,172879821,4,0.833333,0.166667,0.000000,0,1760145,1
3,2,4,0,6,49336.0,214123637,0,0.600000,0.400000,0.000000,0,974651,1
4,2,1,0,3,75152.0,748586141,0,0.629630,0.222222,0.148148,0,275288,1


In [42]:
train_df.info()

print("\nРаспределение target:")
print(train_df['target'].value_counts().sort_index())

print("\nДоли target:")
print(train_df['target'].value_counts(normalize=True).sort_index())

<class 'pandas.DataFrame'>
RangeIndex: 744109 entries, 0 to 744108
Data columns (total 13 columns):
 #   Column                 Non-Null Count   Dtype  
---  ------                 --------------   -----  
 0   item_click_count       744109 non-null  int64  
 1   item_cart_count        744109 non-null  int64  
 2   item_order_count       744109 non-null  int64  
 3   item_total_count       744109 non-null  int64  
 4   item_recency           744109 non-null  float64
 5   item_interaction_span  744109 non-null  int64  
 6   covisitation_score     744109 non-null  int64  
 7   item_click_rate        744109 non-null  float64
 8   item_cart_rate         744109 non-null  float64
 9   item_order_rate        744109 non-null  float64
 10  session                744109 non-null  int64  
 11  aid                    744109 non-null  int64  
 12  target                 744109 non-null  int64  
dtypes: float64(4), int64(9)
memory usage: 73.8 MB

Распределение target:
target
0    723397
1     16672


## Разделение на обучающую и валидационную выборки

Разделять отдельные строки случайным образом нельзя, поскольку одна сессия представлена несколькими товарами-кандидатами.

Все кандидаты одной сессии должны целиком находиться либо в `train`, либо в `validation`. Это сохраняет структуру задачи ранжирования и предотвращает попадание одной и той же сессии в обе выборки.

Используем стандартное разбиние 80:20 (80% train, 20% valid).

In [43]:
from sklearn.model_selection import train_test_split
     
sessions = train_df['session'].unique()

train_sessions, valid_sessions = train_test_split(
    sessions,
    test_size=0.2,
    random_state=55
)

train_data = train_df[train_df['session'].isin(train_sessions)].copy()

valid_data = train_df[train_df['session'].isin(valid_sessions)].copy()

print('Train:', train_data.shape)
print('Validation:', valid_data.shape)
print('Train sessions:', train_data['session'].nunique())
print('Validation sessions:', valid_data['session'].nunique())

Train: (594592, 13)
Validation: (149517, 13)
Train sessions: 8000
Validation sessions: 2000


## Подготовка данных для модели ранжирования

Из обучающей таблицы исключим технические столбцы:

- `session` — используется для определения групп кандидатов;
- `aid` — идентификатор товара;
- `target` — целевая переменная.

Остальные столбцы будут использоваться моделью как признаки.

In [44]:
feature_columns = [
    'item_click_count',
    'item_cart_count',
    'item_order_count',
    'item_total_count',
    'item_recency',
    'item_interaction_span',
    'covisitation_score',
    'item_click_rate',
    'item_cart_rate',
    'item_order_rate'
]

train_data = train_data.sort_values('session').reset_index(drop=True)
valid_data = valid_data.sort_values('session').reset_index(drop=True)

X_train = train_data[feature_columns]
y_train = train_data['target']

X_valid = valid_data[feature_columns]
y_valid = valid_data['target']

train_groups = train_data.groupby('session', sort=False).size().tolist()
valid_groups = valid_data.groupby('session', sort=False).size().tolist()

print('X_train:', X_train.shape)
print('X_valid:', X_valid.shape)

print('Количество train-групп:', len(train_groups))
print('Количество valid-групп:', len(valid_groups))

print('Сумма размеров train-групп:', sum(train_groups))
print('Сумма размеров valid-групп:', sum(valid_groups))

X_train: (594592, 10)
X_valid: (149517, 10)
Количество train-групп: 8000
Количество valid-групп: 2000
Сумма размеров train-групп: 594592
Сумма размеров valid-групп: 149517


## Обучение модели ранжирования

В отличие от обычной модели классификации, модель ранжирования учитывает принадлежность объектов к группам. В нашей задаче одной группой является одна пользовательская сессия, а объектами внутри группы — товары-кандидаты.

Цель модели — научиться присваивать более высокий score товарам с большей релевантностью:

- `0` — товар не появился в future;
- `1` — click;
- `2` — cart;
- `3` — order.


В качестве функции потерь используем `YetiRank` (Loss по умолчанию), предназначенную для обучения моделей ранжирования.

In [45]:
import catboost
print(catboost.__version__)

1.2.10


### CatBoostRanker

Модель получает:

- признаки каждого товара-кандидата;
- его уровень релевантности `target`;
- информацию о том, какие кандидаты относятся к одной сессии.

Во время самого обучения модель будет давать кандидатам `score`, чтобы релевантные товары оказались выше в рамках одной сессии.

In [46]:
from catboost import CatBoostRanker, Pool

train_pool = Pool(
    data=X_train,
    label=y_train,
    group_id=train_data['session']
)

valid_pool = Pool(
    data=X_valid,
    label=y_valid,
    group_id=valid_data['session']
)

## Первая модель CatBoostRanker и оценка ее качества

Обучим базовую модель ранжирования без подбора гиперпараметров.

Цель этого этапа - узнать, насколько хорошо базовая модель ранжирует товары и улучшает качество рекомендаций.

In [47]:
model = CatBoostRanker(
    iterations=500,
    learning_rate=0.05,
    depth=6,
    loss_function='YetiRank',
    eval_metric='NDCG:top=20',
    random_seed=55,
    verbose=50
)

model.fit(
    train_pool,
    eval_set=valid_pool,
    early_stopping_rounds=50,
    verbose=50
)

0:	test: 0.6985615	best: 0.6985615 (0)	total: 95.1ms	remaining: 47.5s
50:	test: 0.7490216	best: 0.7490273 (47)	total: 4.85s	remaining: 42.7s
100:	test: 0.7517833	best: 0.7522228 (87)	total: 9.59s	remaining: 37.9s
150:	test: 0.7526182	best: 0.7532665 (135)	total: 14.3s	remaining: 33s
Stopped by overfitting detector  (50 iterations wait)

bestTest = 0.7532665403
bestIteration = 135

Shrink model to first 136 iterations.


CatBoostRanker(depth=6, eval_metric='NDCG:top=20', iterations=500, learning_rate=0.05, loss_function='YetiRank', random_seed=55, verbose=50)

### Результат обучения базовой модели

Базовый `CatBoostRanker` достиг максимального значения `NDCG@20 ≈ 0.756` на выборке validation.

Лучший результат был на 208 итерации. Последующие 50 итераций значение метрики не повышалось, поэтому сработала ранняя остановка (early_stopping).

Полученное значение NDCG нельзя интерпретировать как долю правильных рекомендаций.

Для оценки качества модели на практике стоит сформировать топ-20 рекомендаций и сранвить их с базовыми подходами.

## Формирование Top-20 рекомендаций

После обучения модель присваивает каждому товару-кандидату значение `score`.

Абсолютное значение `score` само по себе не является вероятностью покупки или клика. Оно используется для определения порядка кандидатов внутри одной сессии.

Для каждой сессии:
1. получаем `score` всех кандидатов;
2. сортируем кандидатов по убыванию `score`;
3. выбираем первые 20 товаров.

В итоге получается рекомендательная система.

In [48]:
valid_data['score'] = model.predict(valid_pool)

valid_ranked = (
    valid_data.sort_values(
        ['session', 'score'],
        ascending=[True, False]
    ).copy()
)

top_20 = (
    valid_ranked.groupby('session', sort=False).head(20).copy()
)

top_20.head(20)

,item_click_count,item_cart_count,item_order_count,item_total_count,item_recency,item_interaction_span,covisitation_score,item_click_rate,item_cart_rate,item_order_rate,session,aid,target,score
0,1,1,0,2,0.000000e+00,9422,10,0.800000,0.200000,0.000000,5,1813405,1,4.907396
30,1,0,0,1,9.294400e+04,0,10,0.823529,0.117647,0.058824,5,63299,0,2.851696
33,1,0,0,1,1.042841e+09,0,10,0.666667,0.333333,0.000000,5,829383,0,0.558807
31,1,0,0,1,9.821768e+08,0,10,1.000000,0.000000,0.000000,5,747242,0,0.487177
34,1,0,0,1,1.042843e+09,0,10,1.000000,0.000000,0.000000,5,1402845,0,0.443462
35,1,0,0,1,1.042844e+09,0,10,1.000000,0.000000,0.000000,5,1808870,0,0.443462
36,1,0,0,1,1.042846e+09,0,10,1.000000,0.000000,0.000000,5,1120175,0,0.443462
37,1,0,0,1,1.042852e+09,0,10,1.000000,0.000000,0.000000,5,342507,0,0.443462
38,1,0,0,1,1.042880e+09,0,10,1.000000,0.000000,0.000000,5,1354785,0,0.443462
39,1,0,0,1,1.042908e+09,0,10,1.000000,0.000000,0.000000,5,1098089,0,0.443462


In [49]:
train_df = train_df.drop(columns=['item_recency:'], errors='ignore')

In [50]:
from pathlib import Path

ARTIFACTS_PATH = Path("../artifacts")
ARTIFACTS_PATH.mkdir(exist_ok=True)

train_df.to_pickle(ARTIFACTS_PATH / "ranking_dataset.pkl")

In [51]:
train_df.to_pickle(ARTIFACTS_PATH / "ranking_dataset.pkl")

## Оценка Top-20 рекомендаций

После получения `score` формируем итоговый список рекомендаций для каждой сессии validation.

Перед расчётом метрик проверим, что для каждой сессии было выбрано не более 20 товаров.

In [52]:
recommendations_per_session = top_20.groupby('session').size()

print(recommendations_per_session.describe())


print('Сессий с более чем 20 рекомендациями:',(recommendations_per_session > 20).sum())

count    2000.000000
mean       18.871000
std         4.295181
min         1.000000
25%        20.000000
50%        20.000000
75%        20.000000
max        20.000000
dtype: float64
Сессий с более чем 20 рекомендациями: 0


### Сохранение релевантных кандидатов в Top-20

Оценим, какая доля релевантных товаров, доступных модели на этапе ранжирования, сохранилась после сокращения списка кандидатов до топ-20.

Расчёт выполняется отдельно для `click`, `cart` и `order`.

In [53]:
target_names = {
    1: 'click',
    2: 'cart',
    3: 'order'
}

for target_value, target_name in target_names.items():
    relevant = valid_data[valid_data['target'] == target_value]
    recommend = top_20[top_20['target'] == target_value]

    recall = (
        len(recommend) / len(relevant)
        if len(relevant) > 0 else 0 
    )

    print(
        f'{target_name}: '
        f'{len(recommend)} / {len(relevant)}'
        f' Recall@20 = {recall:.4f}'
    )

click: 2267 / 3229 Recall@20 = 0.7021
cart: 296 / 372 Recall@20 = 0.7957
order: 341 / 377 Recall@20 = 0.9045


## Оценка End-to-end метрики рекомендательной системы

Для итоговой оценки стоит сравнить топ-20 рекомендаций с реальными действиями пользователь в `future`.

Это позволяет учитывать ошибки обоих этапов:

- candidate generation — релевантный товар мог вообще не попасть в кандидаты;
- ranking — релевантный кандидат мог не попасть в топ-20.

In [54]:
from pathlib import Path

data_path = Path("../data/otto-recsys-train.jsonl")

validation_sessions = set(valid_sessions)

future_targets = {}

with open(data_path, 'r') as file:
    for i, line in enumerate(file):
        if i >= 10000:
            break

        session_data = json.loads(line)
        session_id = session_data['session']

        if session_id not in validation_sessions: # проект train_sessions
            continue

        events = session_data['events']

        future_size = max(1, int(len(events) * 0.25))
        future = events[-future_size:]

        future_targets[session_id] = {
            "click": {
                event["aid"]
                for event in future
                if event["type"] == "clicks"
            },
            "cart": {
                event["aid"]
                for event in future
                if event["type"] == "carts"
            },
            "order": {
                event["aid"]
                for event in future
                if event["type"] == "orders"
            }}

print("Validation sessions:", len(validation_sessions))

print("True sessions:", len(future_targets))

Validation sessions: 2000
True sessions: 2000


### Расчёт end-to-end метрик

In [55]:
import numpy as np

def recall_at_k(recommended, relevant, k=20):
    recommended = recommended[:k]

    if len(relevant) == 0:
        return np.nan

    hits = len(set(recommended) & set(relevant))
    return hits / len(relevant)


def precision_at_k(recommended, relevant, k=20):
    recommended = recommended[:k]

    if len(relevant) == 0:
        return np.nan

    hits = len(set(recommended) & set(relevant))
    return hits / len(recommended) if len(recommended) > 0 else 0


def ndcg_at_k(recommended, relevant, k=20):
    recommended = recommended[:k]

    if len(relevant) == 0:
        return np.nan

    dcg = 0

    for i, aid in enumerate(recommended):
        if aid in relevant:
            dcg += 1 / np.log2(i + 2)

    ideal_hits = min(len(relevant), k)

    idcg = sum(
        1 / np.log2(i + 2)
        for i in range(ideal_hits)
    )

    return dcg / idcg if idcg > 0 else 0

In [56]:
recommendations = (top_20.groupby('session')['aid'].apply(list).to_dict())

results = {}

type_of_interaction = ['click', 'cart', 'order']

for target_name in type_of_interaction:
    recalls = []
    precisions = []
    ndcgs = []

    for session_id in validation_sessions:
        recommend = recommendations.get(session_id, [])
        relevant = future_targets[session_id][target_name]

        if len(relevant) == 0:
            continue 

        recalls.append(recall_at_k(recommend, relevant, k=20))
        precisions.append(precision_at_k(recommend, relevant, k=20))
        ndcgs.append(ndcg_at_k(recommend, relevant, k=20))



    results[target_name] = {
        'Recall@20': np.mean(recalls),
        'Precision@20': np.mean(precisions),
        'NDCG@20': np.mean(ndcgs),
        'Sessions': len(recalls)
    }

results_df = pd.DataFrame(results).T
results_df        

,Recall@20,Precision@20,NDCG@20,Sessions
click,0.234229,0.078890,0.239799,1972.0
cart,0.299141,0.034397,0.221047,658.0
order,0.499461,0.059235,0.409651,305.0


## Сравнение с baseline

Чтобы оценить работу нашей модели ранжирования, сравним ее с простыми эвристическим способом ранжирования, а именно нашим baseline.

В качестве baseline используем `covisitation_score`: товары с большим значением этого признака располагаются выше.

Для корректного сравнения baseline и ML-модель используем:
- одинаковые validation-сессии;
- одинаковый набор кандидатов;
- одинаковое ограничение топ-20.

Сравним результат их работы по одинаковым метрикам.

In [57]:
baseline_ranked = (valid_data.sort_values(['session', 'covisitation_score'],ascending=[True, False]).copy()
)

baseline_top20 = (baseline_ranked.groupby('session', sort=False).head(20).copy()
)

baseline_recommendations = (baseline_top20.groupby('session')['aid'].apply(list).to_dict()
)

len(baseline_recommendations)

2000

Расчитаем метрики для baseline на тех же валидационных выборках.

In [60]:
baseline_results = {}

for target_name in type_of_interaction:
    recalls = []
    precisions = []
    ndcgs = []

    for session_id in validation_sessions:
        recommend = baseline_recommendations.get(session_id, [])
        relevant = future_targets[session_id][target_name]

        if len(relevant) == 0:
            continue

        recalls.append(recall_at_k(recommend, relevant, k=20))
        precisions.append(precision_at_k(recommend, relevant, k=20))
        ndcgs.append(ndcg_at_k(recommend, relevant, k=20))
    baseline_results[target_name] = {
        'Recall@20': np.mean(recalls),
        'Precision@20': np.mean(precisions),
        'NDCG@20': np.mean(ndcgs),
        'Sessions': len(recalls)}

baseline_results_df = pd.DataFrame(baseline_results).T

baseline_results_df

,Recall@20,Precision@20,NDCG@20,Sessions
click,0.149921,0.048870,0.104895,1972.0
cart,0.153493,0.017300,0.072905,658.0
order,0.241374,0.027923,0.123857,305.0


### Сравнение baseline и модели ранжирования

На всех трех метриках модель ранжирования показывает результат выше, чем обычное эвристическое ранжирование по `covisitation_score`.

Наиболее заметны улучшения на типах взаимодействия `cart` и `order`, разница ~ в 2 раза.

In [62]:
feature_importance = pd.DataFrame({
    'feature': feature_columns,
    'importance': model.get_feature_importance(valid_pool)
})

feature_importance = (
    feature_importance
    .sort_values('importance', ascending=False)
    .reset_index(drop=True)
)

feature_importance

,feature,importance
0,item_recency,0.089578
1,item_cart_count,0.007964
2,item_interaction_span,0.006041
3,item_click_count,0.004216
4,covisitation_score,0.003610
5,item_total_count,0.002553
6,item_order_count,0.000645
7,item_click_rate,0.000461
8,item_cart_rate,0.000319
9,item_order_rate,0.000010


### Анализ важности признаков

Наиболее важным признаком оказался `item_recency` - давность последнего взаимодействия пользователя с товаром. Его вклад в модель наиболее весомый по сравнению с другими признаками.

Следующими по важности оказались:

- `item_cart_count` — количество добавлений товара в корзину;
- `item_interaction_span` — временной диапазон взаимодействий с товаром;
- `item_click_count` — количество кликов по товару;
- `covisitation_score` — сила связи товара с другими товарами из истории сессии.

При этом самыми незначительными признаками оказались `item_order_count`, `item_click_rate`, `item_cart_rate`, `item_order_rate`.

## Анализ ошибок модели

После оценки общего качества модели рассмотрим, в каких типах сессий рекомендации работают наиболее хорошо или плохо.

В первую очередь проверим зависимость качества рекомендаций от количества кандидатов в сессии. Это позволит понять, меняется ли качество модели в зависимости от сложности задачи ранжирования.

Посчитаем **NDCG@20** для каждой сессии.

In [64]:
session_errors = []

for session_id in validation_sessions:
    relevant = future_targets[session_id]['order']

    if len(relevant) == 0:
        continue

    recommended = recommendations.get(session_id, [])
    ndcg = ndcg_at_k(recommended, relevant, k=20
    )

    candidate_count = len(valid_data[valid_data['session'] == session_id])

    session_errors.append({
        'session': session_id,
        'candidate_count': candidate_count,
        'ndcg_order': ndcg
    })

session_errors_df = pd.DataFrame(session_errors)

session_errors_df.head()

,session,candidate_count,ndcg_order
0,35,100,0.000000
1,36,100,0.296082
2,52,100,0.469279
3,53,89,0.722727
4,8279,100,0.833520


In [65]:
session_errors_df['candidate_count'].describe()

count    305.000000
mean      93.334426
std       18.118932
min        2.000000
25%      100.000000
50%      100.000000
75%      100.000000
max      100.000000
Name: candidate_count, dtype: float64

Медиана - 100, в то время как среднее значение - 93.334, значит в выборке есть сессии с неполным пулом кандидатов.

Сейчас мы хотим проверить гипотезу, что качества ранжирования в сессиях из неполного пула кандидатов (меньше 100) будет показывать себя хуже, чем с полным пулом.

In [69]:
session_errors_df['candidate_group'] = '100'

session_errors_df.loc[session_errors_df['candidate_count'] < 100, 'candidate_group'] = '< 100'

candidate_analysis = session_errors_df.groupby('candidate_group').agg(
    sessions=('session', 'count'),
    mean_candidates=('candidate_count', 'mean'),
    mean_ndcg=('ndcg_order', 'mean')
)
candidate_analysis

,sessions,mean_candidates,mean_ndcg
candidate_group,,,
100,252,100.000000,0.359686
< 100,53,61.641509,0.647223


## Проверка гипотезы

По полученным результатам **NDCG@20** для сессий с пулом кандидатов <100 оказался даже, чем в полном пуле (0.647 против 0.360). Получается, что наши прогнозы были неверны и гипотеза не подтвердилась. На текущей выборке модель лучше ранжирует товары в сессиях с меньшим количеством кандидатов.

При этом нельзя уверенно сказать, что причинный эффект размера пула кандидатов повлиял на результат, потому что группы в выборке немного не сбалансированы (252 против 53), такие сессии могут отличаться по другим характеристикам.

### Зависимость качества от длины истории сессии

Далее стоит проверить, зависит ли качество ранжирования от количества собыйий в сесиии.

Проверяем гипотезу, что модель на более длинных сессиях модель покажет результат лучше, потому что у нее банально будет больше информации о поведении пользователя, так как в таких сессиях будет больше взаимодействия с товаром.

Проверим эту гипотезу на validation-сессиях с целевым действием `order`.

In [73]:
history_lengths = (
    valid_data
    .groupby('session')['item_total_count']
    .sum()
)

session_errors_df['history_length'] = (
    session_errors_df['session'].map(history_lengths)
)

session_errors_df[
    ['session', 'history_length', 'ndcg_order']
].head()

,session,history_length,ndcg_order
0,35,174,0.000000
1,36,160,0.296082
2,52,174,0.469279
3,53,84,0.722727
4,8279,39,0.833520


In [72]:
session_errors_df['history_length'].describe()

count    305.000000
mean      88.727869
std       63.833521
min        3.000000
25%       39.000000
50%       73.000000
75%      136.000000
max      302.000000
Name: history_length, dtype: float64

Распределение сессий достаточно широкое: от 3 до 302 взаимодействий, медиана составляет 73.

Для анализа разделим сессии на четыре группы по квартилям распределения:

- короткие — до 39 взаимодействий;
- средние — от 40 до 73;
- длинные — от 74 до 136;
- очень длинные — более 136.

Такое разбиение позволит получить группы примерно равного размера, что поможет сравнить качество ранжирования.

In [75]:
session_errors_df['history_group'] = 'очень длинные'

session_errors_df.loc[session_errors_df['history_length'] <= 136, 'history_group'] = 'длинные'

session_errors_df.loc[session_errors_df['history_length'] <= 73, 'history_group'] = 'средние'

session_errors_df.loc[session_errors_df['history_length'] <= 39, 'history_group'] = 'короткие'

In [76]:
history_analysis = session_errors_df.groupby('history_group').agg(
    sessions=('session', 'count'),
    mean_history_length=('history_length', 'mean'),
    mean_ndcg=('ndcg_order', 'mean'))



history_analysis

,sessions,mean_history_length,mean_ndcg
history_group,,,
длинные,75,99.693333,0.317506
короткие,79,21.000000,0.611805
очень длинные,76,181.407895,0.236923
средние,75,55.186667,0.463894


### Проверка гипотезы №2

Полученные результаты опровергли исходную гипотезу, так как в выборке прослеживается тенденция, что с уменьшением количества взаимодействий увеличивается и метрика **NDCG@20**. 

Такой результат можно связать с проверкой предыдущей гипотезы, где мы узнали, что сессии с меньшим пулом кандидатов тоже показали более высокие метрики. Возможно, что они отражают общую закономерность: модель лучше работает с менее сложными сессиями, где меньше взаимодействия товаров и, вследствие чего, меньше неопределенности.

Таким образом, данная модель лучше ранжирует товары в сессиях с короткой историей.

## Сохранение модели

После обучения и оценки сохраним модель ранжирования.

Это позволит использовать обученную модель для построения рекомендаций без необходимости повторного обучения.

In [78]:
from pathlib import Path

ARTIFACTS_PATH = Path('../artifacts')
ARTIFACTS_PATH.mkdir(exist_ok=True)

model.save_model(ARTIFACTS_PATH / 'catboost_ranker.cbm')

The history saving thread hit an unexpected error (OperationalError('attempt to write a readonly database')).History will not be written to the database.
